# RetailPulse — Sales Analysis
Recreates and extends the source visualizations using the normalized analytical view.


In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
import plotly.express as px

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
conn = sqlite3.connect(ROOT / 'data' / 'customer_sales.db')
facts = pd.read_sql_query('SELECT * FROM vw_sales_facts', conn, parse_dates=['order_date'])


In [ ]:
total_sales = facts['line_revenue'].sum()
orders = facts['order_id'].nunique()
customers = facts['customer_id'].nunique()
{'sales': total_sales, 'orders': orders, 'customers': customers, 'aov': total_sales/orders}


In [ ]:
monthly = facts.assign(month=facts.order_date.dt.to_period('M').dt.to_timestamp()).groupby('month', as_index=False).agg(sales=('line_revenue','sum'), orders=('order_id','nunique'))
px.line(monthly, x='month', y='sales', markers=True, title='Monthly Sales')


In [ ]:
by_customer = facts.groupby('customer_name', as_index=False).agg(total_sales=('line_revenue','sum')).nlargest(15,'total_sales').sort_values('total_sales')
px.bar(by_customer, x='total_sales', y='customer_name', orientation='h', title='Top Customers by Sales')


In [ ]:
by_state = facts.groupby('state', as_index=False).agg(total_sales=('line_revenue','sum')).sort_values('total_sales', ascending=False)
px.bar(by_state, x='state', y='total_sales', title='Sales by State')


In [ ]:
by_product = facts.groupby('product_name', as_index=False).agg(total_sales=('line_revenue','sum')).nlargest(10,'total_sales').sort_values('total_sales')
px.bar(by_product, x='total_sales', y='product_name', orientation='h', title='Top 10 Products')
